Notebook to showcase <src/pedsilicoICH/insert_lesion_3D.py>
Written 8/26/24, Jayse M. Weaver

In [1]:
# %pip install monai --upgrade --force

In [2]:
%pip install ipywidgets -q

Note: you may need to restart the kernel to use updated packages.


In [16]:
# general set up

from pathlib import Path
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import skimage as ski
import sys
from utils import scrollview

# import custom modules
#from insert_lesion import *

# options
save_output = False
plot_final = False
source = 'MIDA'

# custom colormap to showcase masks
yellow, norm = mpl.colors.from_levels_and_colors(levels=[0, 1], colors=['blue', 'yellow'], extend='max')

First, the phantom of choice must be pre-processed before calling <insert_lesion_3D> as the lesion insertion pipeline is phantom-agnostic.
The lesion insertion code requires:
* header: NifTI header information
* volume: full image volume
* boundary: binary mask of brain's edge, from existing atlas (MIDA) or brain mask segmentation (NIHPD)
* skull: binary skull mask
* init_slice: center slice for generated hemorrhage
* hematoma_type: subdural or epidural

In [17]:
if source == 'MIDA': # MIDA phantom set up:
    mida_path = Path('/home/jayse.weaver/the_lab/MIDA_v1.0/MIDA_v1_voxels') # define path to phantom
    img = nib.load(mida_path/'MIDA_v1.nii') # load nifti 
    atlas = img.get_fdata().transpose(0, 2, 1) # transpose volume to [x, y, slice]
    dura_map = np.zeros_like(atlas)
    dura_map[np.where(atlas == 1.0)] = 1.0 # isolate dura voxels to separate boundary mask

    # Identify all voxels outside of dura to use as anchor points for warping
    # 43, 51-54, and 61-64 are all skull, muscle, subcutaneous tissue, etc
    skull_map = np.zeros_like(atlas)
    #skull_tissue_map[np.where(axial_slice == 43)] = 1.0
    # skull_tissue_map[np.where((axial_slice >= 51) & (axial_slice <= 54))] = 1.0
    # skull_tissue_map[np.where((axial_slice >= 61) & (axial_slice <= 64))] = 1.0
    skull_map[np.where((atlas == 53))] = 1.0 # for now, just do skull

    slice_num = 350

    df = pd.read_csv(mida_path/'MIDA_v1.csv')

elif source == 'NIHPD': # NIHPD Objective 1 Atlases set up:
    nihpd_path = str(Path('../NIHPD_Obj1'))
    age_range = '04.5-08.5'
    img = nib.load(nihpd_path + '/nihpd_asym_'+age_range+'_csf.nii')
    csf = nib.load(nihpd_path + '/nihpd_asym_'+age_range+'_csf.nii').get_fdata().transpose(1, 0, 2)[::-1, :, :]
    wm = nib.load(nihpd_path + '/nihpd_asym_'+age_range+'_wm.nii').get_fdata().transpose(1, 0, 2)[::-1, :, :]
    gm = nib.load(nihpd_path + '/nihpd_asym_'+age_range+'_gm.nii').get_fdata().transpose(1, 0, 2)[::-1, :, :]
    mask = nib.load(nihpd_path + '/nihpd_asym_'+age_range+'_mask.nii').get_fdata().transpose(1, 0, 2)[::-1, :, :]
    pdw = nib.load(nihpd_path + '/nihpd_asym_'+age_range+'_pdw.nii').get_fdata().transpose(1, 0, 2)[::-1, :, :]

    # need to use pdw (or t1w) to get rudimentary skull mask:
    skull_map = (mask == 0)*pdw / pdw.max()
    skull_map[skull_map < 0.1] = 0

    threshold = 0.50
    atlas = np.zeros_like(wm)
    atlas[np.where(csf > 0)] = 15
    atlas[np.where(wm > 0)] = 20
    atlas[np.where(gm > 0.5)] = 45
    atlas[np.where(skull_map > 0)] = 1000
    atlas[atlas == 0] = -1000

    dura_map = ski.segmentation.find_boundaries(mask, mode='inner', background=0)

    slice_num = 100

else:
    sys.exit("Iimage source not specified (try 'MIDA' or 'NIHPD')")

In [13]:
plt.imshow(mask)

NameError: name 'mask' is not defined

Call lesion insertion function

In [10]:
!cat ~/PedSilicoICH/src/pedsilicoICH/__init__.py
from pedsilicoICH.insert_lesion_3D import insert_dural_3D

# need to add # of slices for hemorrhage or make it a pseudorandom option
header = img.header
spacing = header['pixdim'][1:4]
hemorrhage_mask = insert_dural_3D(spacing, volume=atlas, boundary=dura_map, init_slice=slice_num,
                          hematoma_type='epidural', verbose=True, plot_opt=plot_final)

print(hemorrhage_mask.shape)

HU_volume = np.ones_like(atlas)
if source == 'MIDA':
    for idx, row in df.iterrows():
        HU_volume[atlas==row['MIDA_ID']] = row['HU']
    HU_volume[hemorrhage_mask==1] = 70 # set lesion to 60 HU

# plt.subplot(1,3,1)
# plt.imshow(HU_volume[:, :, 345], vmin=-40, vmax=120)
# plt.subplot(1,3,2)
# plt.imshow(HU_volume[:, :, 350], vmin=-40, vmax=120)
# plt.subplot(1,3,3)
# plt.imshow(HU_volume[:, :, 355], vmin=-40, vmax=120)
# plt.colorbar()

if save_output:
    # save
    # transpose back to original slice ordering
    #img_array[:, :, 349][final_hemorrhage] = 117
    #img_array[:, :, 35] = new_volume
    save_img = nib.Nifti1Image(hemorrhage_mask.transpose(0, 1, 2), img.affine)
    nib.save(save_img, 'test_hemorrhage_mask.nii')

    HU_img = nib.Nifti1Image(HU_volume.transpose(0, 1, 2), img.affine)
    nib.save(HU_img, 'test_HU_hemorrhage.nii')

cat: /home/brandon.nelson/PedSilicoICH/src/pedsilicoICH/__init__.py: No such file or directory


TypeError: insert_dural_3D() missing 2 required positional arguments: 'verbose' and 'plot_opt'

Visualize results (for demo, scroll to slices 345-355)

In [9]:
scrollview(HU_volume.transpose(2, 0, 1))

interactive(children=(IntSlider(value=240, description='idx', max=479), Output()), _dom_classes=('widget-inter…